
# 11 — Prompt Optimization and Evaluation

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Inspired by course notebooks:**
- `prompt_optimization.ipynb`
- `few_shot_learning.ipynb`

## Goal

Compare two prompt variants for extracting structured
information from customer-support messages.

The notebook covers:

- Defining a baseline and an improved prompt.
- Using few-shot examples to demonstrate the desired format.
- Testing both prompts on the same inputs.
- Parsing and validating JSON responses.
- Measuring extraction accuracy against known answers.
- Comparing results without an additional LLM evaluator.

**Adaptation:** The course demonstrates prompt A/B testing
and human feedback on generated product names. This version
uses structured information extraction and deterministic
Python evaluation.

**API usage:** Offline preparation and evaluation require
zero requests. The live experiment makes four Gemini
requests in total, if each of the four test runs is attempted.

No real customer data is used.



## 1. Why Evaluate Prompts?

Different instructions can produce different answers
from the same language model.

A prompt that sounds more detailed is not automatically
better. We need to test its results.

We will compare:

**Prompt A — Baseline**

A short instruction asking the model to extract fields
from a customer message.

**Prompt B — Improved**

A more specific instruction with field definitions,
output rules and few-shot examples.

Both prompts will process the same messages using
the same Gemini model.

Python will compare the model's responses against
manually defined correct answers.

This is a small controlled demonstration, not a
statistically reliable benchmark.


In [1]:

TEST_CASES = [
    {
        "id": "case_1",
        "message": (
            "Hi, my two Wireless Earbuds from order "
            "1001 arrived damaged."
        ),
        "expected": {
            "order_id": "1001",
            "product": "Wireless Earbuds",
            "quantity": 2,
            "delivery_issue": "damaged"
        }
    },
    {
        "id": "case_2",
        "message": (
            "Order #2042: I ordered one Bluetooth Speaker, "
            "but it hasn't arrived and the delivery "
            "estimate was yesterday."
        ),
        "expected": {
            "order_id": "2042",
            "product": "Bluetooth Speaker",
            "quantity": 1,
            "delivery_issue": "late"
        }
    }
]

REQUIRED_FIELDS = [
    "order_id",
    "product",
    "quantity",
    "delivery_issue"
]

for case in TEST_CASES:
    print("\n", case["id"])
    print("Message:", case["message"])
    print("Expected:", case["expected"])



 case_1
Message: Hi, my two Wireless Earbuds from order 1001 arrived damaged.
Expected: {'order_id': '1001', 'product': 'Wireless Earbuds', 'quantity': 2, 'delivery_issue': 'damaged'}

 case_2
Message: Order #2042: I ordered one Bluetooth Speaker, but it hasn't arrived and the delivery estimate was yesterday.
Expected: {'order_id': '2042', 'product': 'Bluetooth Speaker', 'quantity': 1, 'delivery_issue': 'late'}



## 2. Design Two Prompt Variants

The baseline and improved prompts have the same task:
extract four fields from a customer message.

The difference is the level of guidance.

**Prompt A** gives a brief request for JSON extraction.

**Prompt B** provides:
- Exact field names.
- Defined delivery-issue categories.
- Rules for missing information.
- Examples of correctly formatted answers.

The examples demonstrate the task without including
either of our two evaluation messages.

This prevents copying a test answer directly from
the few-shot examples.


In [2]:

PROMPT_A = """
Extract the order ID, product, quantity and delivery issue
from the customer message.

Return the information as JSON.
""".strip()


PROMPT_B = """
You extract structured data from customer-support messages.

Return ONLY a valid JSON object with exactly these fields:
- order_id: string or null
- product: string or null
- quantity: integer or null
- delivery_issue: one of "damaged", "late",
  "missing", "none"

Rules:
- Do not invent information.
- Use null for missing order IDs, products or quantities.
- Use "damaged" if an item arrived damaged.
- Use "late" if delivery is overdue.
- Use "missing" if an expected item is absent
  from an otherwise received order.
- Use "none" if no delivery problem is described.
- Return no explanations or Markdown formatting.

Examples:

Customer: Order 9000 contained three Travel Headphones,
but one was missing from the package.
Output:
{"order_id":"9000","product":"Travel Headphones",
"quantity":3,"delivery_issue":"missing"}

Customer: Are the Portable Chargers available?
Output:
{"order_id":null,"product":"Portable Chargers",
"quantity":null,"delivery_issue":"none"}
""".strip()


PROMPTS = {
    "baseline": PROMPT_A,
    "improved": PROMPT_B
}

print("Baseline prompt length:", len(PROMPT_A))
print("Improved prompt length:", len(PROMPT_B))
print("Prompt variants ready.")
print("Gemini API requests: 0")


Baseline prompt length: 118
Improved prompt length: 971
Prompt variants ready.
Gemini API requests: 0



## 3. Deterministic Evaluation

Instead of asking another AI model which response
is better, Python will evaluate the output.

We will measure:

**Valid JSON**
Can the response be parsed as a JSON object?

**Schema validity**
Does it contain exactly the required fields with
the expected data types and allowed category values?

**Field accuracy**
How many extracted values match the reference labels?

**Exact-match accuracy**
Does the entire response match the expected answer?

For this small experiment, each field has equal weight.

Missing, malformed or incorrect values receive no
credit for that field.


In [3]:

import json


def parse_json_response(text):
    if not isinstance(text, str) or not text.strip():
        return None

    cleaned = text.strip()

    # Accept an optional Markdown JSON code fence.
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        if len(lines) >= 3 and lines[-1].strip() == "```":
            cleaned = "\n".join(lines[1:-1]).strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        return None

    return parsed if isinstance(parsed, dict) else None


def valid_schema(data):
    if not isinstance(data, dict):
        return False

    if set(data.keys()) != set(REQUIRED_FIELDS):
        return False

    if data["order_id"] is not None and not isinstance(
        data["order_id"], str
    ):
        return False

    if data["product"] is not None and not isinstance(
        data["product"], str
    ):
        return False

    quantity = data["quantity"]

    if quantity is not None and (
        type(quantity) is not int or quantity < 1
    ):
        return False

    if data["delivery_issue"] not in {
        "damaged", "late", "missing", "none"
    }:
        return False

    return True


def evaluate_response(text, expected):
    predicted = parse_json_response(text)
    schema_ok = valid_schema(predicted)

    # Require valid JSON and the complete schema before
    # awarding any field accuracy.
    field_results = {}

    for field in REQUIRED_FIELDS:
        field_results[field] = (
            schema_ok and predicted[field] == expected[field]
        )

    correct_fields = sum(field_results.values())

    return {
        "json_valid": predicted is not None,
        "schema_valid": schema_ok,
        "correct_fields": correct_fields,
        "total_fields": len(REQUIRED_FIELDS),
        "field_accuracy": (
            correct_fields / len(REQUIRED_FIELDS)
        ),
        "exact_match": (
            schema_ok and correct_fields == len(REQUIRED_FIELDS)
        ),
        "predicted": predicted,
        "field_results": field_results
    }


print("Evaluation functions ready.")


Evaluation functions ready.


In [4]:

correct_example = json.dumps(TEST_CASES[0]["expected"])

correct_score = evaluate_response(
    correct_example,
    TEST_CASES[0]["expected"]
)

assert correct_score["schema_valid"] is True
assert correct_score["field_accuracy"] == 1.0
assert correct_score["exact_match"] is True


incorrect_example = json.dumps({
    "order_id": "1001",
    "product": "Wireless Earbuds",
    "quantity": 1,
    "delivery_issue": "damaged"
})

incorrect_score = evaluate_response(
    incorrect_example,
    TEST_CASES[0]["expected"]
)

assert incorrect_score["field_accuracy"] == 0.75
assert incorrect_score["exact_match"] is False


invalid_example = '{"order_id": "1001"'

invalid_score = evaluate_response(
    invalid_example,
    TEST_CASES[0]["expected"]
)

assert invalid_score["json_valid"] is False
assert invalid_score["field_accuracy"] == 0.0

print("All offline evaluation tests passed.")
print("Gemini API requests: 0")


All offline evaluation tests passed.
Gemini API requests: 0



## 4. Live Prompt Comparison

Both prompt variants are evaluated on the same two
customer messages using Gemini 3.5 Flash Lite.

The model and temperature settings remain consistent
across both variants to support a controlled comparison.

Each experiment is executed in a separate notebook cell,
allowing individual responses to be inspected and preserved.

Successful raw responses are stored for reuse within the
active Colab runtime, while automatic retries are disabled.

The experiment focuses on comparing structured output
quality across the two prompt designs.

Because the evaluation contains only two examples,
its results demonstrate observed behaviour rather
than statistically generalizable performance.


In [5]:

from openai import OpenAI
from getpass import getpass

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=0,
    timeout=120.0
)

safe_client = client.with_options(
    max_retries=0,
    timeout=120.0
)

print("Selected model:", MODEL)
print("Client ready. No API requests made.")


Enter your Gemini API key: ··········
Selected model: gemini-3.5-flash-lite
Client ready. No API requests made.


In [6]:

# Store successful responses in the active Colab runtime.
if "experiment_results" not in globals():
    experiment_results = {}


def run_experiment(variant, case_index):
    if variant not in PROMPTS:
        raise ValueError("Unknown prompt variant.")

    if not 0 <= case_index < len(TEST_CASES):
        raise ValueError("Invalid case index.")

    case = TEST_CASES[case_index]
    result_key = (variant, case["id"])

    # Never repeat a previously successful request
    # in this runtime.
    if result_key in experiment_results:
        print("Using previously saved result:", result_key)
        return experiment_results[result_key]

    response = safe_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": PROMPTS[variant]
            },
            {
                "role": "user",
                "content": case["message"]
            }
        ],
        temperature=0
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError("Gemini returned empty content.")

    # Save the response BEFORE evaluating it.
    # Even malformed JSON is useful evidence.
    experiment_results[result_key] = {
        "variant": variant,
        "case_id": case["id"],
        "raw_response": content
    }

    print("Completed:", result_key)
    print("Response:\n", content)

    return experiment_results[result_key]


print("Experiment runner ready.")
print("No API requests made.")


Experiment runner ready.
No API requests made.


In [7]:

baseline_case_1 = run_experiment("baseline", 0)


Completed: ('baseline', 'case_1')
Response:
 ```json
{
  "order_id": "1001",
  "product": "Wireless Earbuds",
  "quantity": 2,
  "delivery_issue": "arrived damaged"
}
```


In [8]:

baseline_case_2 = run_experiment("baseline", 1)


Completed: ('baseline', 'case_2')
Response:
 ```json
{
  "order_id": "2042",
  "product": "Bluetooth Speaker",
  "quantity": 1,
  "delivery_issue": "Hasn't arrived, past the delivery estimate of yesterday"
}
```


In [9]:

improved_case_1 = run_experiment("improved", 0)


Completed: ('improved', 'case_1')
Response:
 {"order_id":"1001","product":"Wireless Earbuds","quantity":2,"delivery_issue":"damaged"}


In [10]:

improved_case_2 = run_experiment("improved", 1)


Completed: ('improved', 'case_2')
Response:
 {"order_id":"2042","product":"Bluetooth Speaker","quantity":1,"delivery_issue":"late"}



## 5. Compare Prompt Performance

After collecting the model responses, evaluate both
prompt variants with the same deterministic scoring logic.

We will calculate:

- JSON validity.
- Schema validity.
- Number of correct extracted fields.
- Field-level accuracy.
- Number of completely correct responses.

The evaluation does not require Gemini.

We will also display results for individual test cases
so that overall scores do not hide specific errors.

The metrics describe performance only on the two
synthetic test cases in this notebook.


In [11]:

scored_results = []

for variant in ["baseline", "improved"]:
    for case in TEST_CASES:
        key = (variant, case["id"])

        if key not in experiment_results:
            print("Missing result:", key)
            continue

        raw = experiment_results[key]["raw_response"]

        score = evaluate_response(
            raw,
            case["expected"]
        )

        row = {
            "variant": variant,
            "case_id": case["id"],
            **score
        }

        scored_results.append(row)

        print(
            f"{variant:8} {case['id']:6} "
            f"JSON={score['json_valid']} "
            f"Schema={score['schema_valid']} "
            f"Fields={score['correct_fields']}/4 "
            f"Exact={score['exact_match']}"
        )

print("\nScored responses:", len(scored_results))
print("Expected responses:", 4)
print("Additional Gemini API requests: 0")


baseline case_1 JSON=True Schema=False Fields=0/4 Exact=False
baseline case_2 JSON=True Schema=False Fields=0/4 Exact=False
improved case_1 JSON=True Schema=True Fields=4/4 Exact=True
improved case_2 JSON=True Schema=True Fields=4/4 Exact=True

Scored responses: 4
Expected responses: 4
Additional Gemini API requests: 0


In [12]:

summary = {}

for variant in ["baseline", "improved"]:
    rows = [
        row for row in scored_results
        if row["variant"] == variant
    ]

    correct_fields = sum(
        row["correct_fields"] for row in rows
    )

    possible_fields = sum(
        row["total_fields"] for row in rows
    )

    summary[variant] = {
        "completed_cases": len(rows),
        "correct_fields": correct_fields,
        "possible_fields": possible_fields,
        "field_accuracy": (
            correct_fields / possible_fields
            if possible_fields else None
        ),
        "exact_matches": sum(
            row["exact_match"] for row in rows
        )
    }

for variant, metrics in summary.items():
    print(f"\n{variant.upper()}")
    print("Completed cases:", metrics["completed_cases"])
    print(
        "Correct fields:",
        f"{metrics['correct_fields']}/"
        f"{metrics['possible_fields']}"
    )
    print("Field accuracy:", metrics["field_accuracy"])
    print("Exact matches:", metrics["exact_matches"])

complete = all(
    metrics["completed_cases"] == len(TEST_CASES)
    for metrics in summary.values()
)

print("\nExperiment complete:", complete)

if complete:
    a = summary["baseline"]["field_accuracy"]
    b = summary["improved"]["field_accuracy"]

    if b > a:
        print("Improved prompt performed better.")
    elif a > b:
        print("Baseline prompt performed better.")
    else:
        print("Both prompts achieved equal field accuracy.")
else:
    print(
        "Comparison incomplete. Do not declare a winner."
    )



BASELINE
Completed cases: 2
Correct fields: 0/8
Field accuracy: 0.0
Exact matches: 0

IMPROVED
Completed cases: 2
Correct fields: 8/8
Field accuracy: 1.0
Exact matches: 2

Experiment complete: True
Improved prompt performed better.



## Findings and Limitations

### Experiment Design

- Created two prompt variants for the same extraction task.
- Used identical synthetic test messages for both prompts.
- Defined expected answers before making model requests.
- Tested JSON parsing, schema validity and field accuracy.
- Used deterministic Python scoring instead of an LLM judge.
- Kept model settings consistent across the two variants.

### Prompting Techniques

**Baseline prompting:** A short instruction requests
structured information without detailed output rules.

**Few-shot prompting:** The improved prompt includes
examples showing how inputs should map to outputs.

**Explicit constraints:** The improved prompt defines
field names, allowed values and missing-data behavior.

### Evaluation Limitations

- Only two synthetic test cases are used.
- Results from such a small sample cannot establish
  statistically reliable prompt superiority.
- The evaluation uses strict string matching.
- All fields have equal weight in the reported accuracy.
- Schema-invalid responses receive zero field credit.
- The improved prompt is longer and uses more input tokens.
- Few-shot examples and explicit constraints change
  together, so their individual effects cannot be isolated.
- Model responses can vary across repeated executions.
- The experiment does not measure production error rates
  or independently verify real customer data.

### Observed Results

- All four Gemini requests completed successfully.
- Both baseline responses contained valid JSON, but their
  `delivery_issue` values did not meet the required schema.
- Under the strict schema-first scoring policy, the
  baseline achieved 0/8 correct fields and 0/2 exact matches.
- The improved prompt achieved 8/8 correct fields,
  valid schemas for both cases, and 2/2 exact matches.
- The baseline nevertheless extracted the correct order IDs,
  product names and quantities; its zero score reflects
  the strict schema-validation rule.
- The improved prompt demonstrated better structured-output
  compliance on the two evaluation examples.

### Conclusion

This notebook demonstrates prompt optimization as a
measurable experiment rather than a subjective judgment
about which instructions sound better.

By testing prompt variants on the same labeled examples
and evaluating outputs with deterministic Python rules,
we can compare formatting reliability and extraction
accuracy without additional evaluator model requests.

The observed results should be interpreted only within
the small synthetic dataset used in this exercise.
